# Human-AI complementarity

Example notebook for the BrainGPT project

## Load libraries

In [ ]:
import os
import torch
import logging
import submitit

import numpy as np
import pandas as pd

from haico.Bayesian_HM_model import BayesianCombinationModel

In [ ]:
def setup_logging(logsdir="logs"):
    # get the hostname
    hostname = os.uname().nodename
    
    if 'SUBMITIT_EXECUTOR' in os.environ:
        logger = logging.getLogger("submitit") # using submitit task logger
        print(f'using submitit logger at {hostname}')
    else :
        # using hostname as the logger name
        logger = logging.getLogger(hostname)
        logger.setLevel(logging.DEBUG)
    
    # avoid duplicated handlers (duplicated log messages)
    if logger.hasHandlers():
        return logger

    if not os.path.exists(logsdir):
        os.makedirs(logsdir)

    # today date
    import datetime

    today = datetime.datetime.now().strftime("%Y_%m_%d")
    logfile = os.path.join(logsdir, "output_{}.log".format(today))
    formatter = logging.Formatter("{levelname} [{name}]: {asctime} - {message}", style="{")

    # Console handler
    chandler = logging.StreamHandler()
    chandler.setLevel(logging.DEBUG)
    chandler.setFormatter(formatter)
    logger.addHandler(chandler)

    # File handler
    fhandler = logging.FileHandler(logfile, "a")
    fhandler.setLevel(logging.DEBUG)
    fhandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.info(f"Logging to: {logfile}")

    return logger

# get the parent directory of the current directory
# parentdir = os.path.dirname(os.getcwd())

# using logger to log messages on the console and in a file
logsdir = os.path.join(os.getcwd(), "logs-brainbench")
logger = setup_logging(logsdir)

In [ ]:
# Set root directory path
root_path = os.path.join(os.getcwd(), "..")
root_path = os.path.abspath(root_path)
print("Root path:", root_path)

# Set path to the data directory
data_path = os.path.join(root_path, 'data')
print("Data path:", data_path)

# Set a random seed for PyTorch
seed = 0
torch.manual_seed(seed)

# If using CUDA, set the random seed for CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

## Define parameters

In [ ]:
selected_LLMs = ['meta-llama--Llama-2-7b-chat-hf', 'meta-llama--Llama-2-13b-chat-hf', 'meta-llama--Llama-2-70b-chat-hf']

# Set parameters for the Bayesian model
num_samples, warmup_steps, num_chains = 50, 1000, 8

results_path = f"HM_sd0_num_samples_{num_samples}_warmup_steps_{warmup_steps}_num_chains_{num_chains}/"

# Set path to the results directory
results_path = os.path.join(root_path, 'results_brainbench', results_path)
os.makedirs(results_path, exist_ok=True)
print("Results will be saved to:", results_path)

# Abstract type to be utilized
abstract_type = ['human', 'machine'][1]

## Prepare human-machine data

In [ ]:
# Read human data
abstract_folder = f"{abstract_type if abstract_type == 'human' else 'llm'}_abstracts"
online_study = pd.read_csv(os.path.join(data_path, 'human/wide_excluded_data.csv'))

# Analyze data by abstract type
abstract_idx = online_study['journal_section'].str.startswith(abstract_type)
online_study = online_study[abstract_idx]

In [ ]:
# Get ground truth labels
order_labels = np.load(f"{data_path}/machine/{selected_LLMs[0]}/{abstract_folder}/labels.npy")

In [ ]:
classification = pd.DataFrame()
classification.loc[:,'abstract_id'] = np.array([np.where(np.unique(online_study['abstract_id'])==i)[0][0] for i in online_study['abstract_id']])
confidence = classification.copy()

classification = classification.merge(pd.DataFrame(order_labels, columns=['true labels']) , left_on='abstract_id', right_index=True)

classification.loc[:,'Human'] = np.array([j if i == 1 else 1 - j for i, j in zip(online_study['correct'], classification['true labels'])])
confidence.loc[:,'Human'] = np.where(online_study['confidence'].values > 66, 2,
                                     np.where(online_study['confidence'].values <= 33, 0, 1)) 

classification.sort_values(by='abstract_id', inplace=True)
confidence.sort_values(by='abstract_id', inplace=True)

classification.reset_index(drop=True, inplace=True)
confidence.reset_index(drop=True, inplace=True)

In [ ]:
def softmax(x, axis=None):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

In [ ]:
for i in selected_LLMs:
    # Read PPL scores of machine classifier
    machine_PPL = np.load(f"{data_path}/machine/{i}/{abstract_folder}/PPL_A_and_B.npy")
    # Get classification results
    machine_name = i.lstrip('meta-llama--Llama-2-').rstrip('-chat-hf').upper()
    machine_classification = pd.DataFrame(np.argmin(machine_PPL, axis=1), columns=[machine_name])
    classification = classification.merge(machine_classification, left_on='abstract_id', right_index=True)
    # Define confidence as PPL difference
    machine_confidence = pd.DataFrame(softmax(machine_PPL, axis=1), columns=[machine_name+'-A',machine_name+'-B'])
    confidence = confidence.merge(machine_confidence, left_on='abstract_id', right_index=True)

## Run Bayesian combination model

In [ ]:
# Define process cross_validation_fold as a function
def process_cross_validation_fold(fold, machine_clf, N):
    # Train/test data for current fold
    machine_probscores_train = torch.from_numpy(confidence[confidence['abstract_id']!=fold][[machine_clf+'-A',machine_clf+'-B']].values)
    human_classification_train = torch.from_numpy(classification[classification['abstract_id']!=fold]['Human'].values).to(torch.int64)
    human_confidence_train = torch.from_numpy(confidence[confidence['abstract_id']!=fold]['Human'].values)
    truelabel_train = torch.from_numpy(classification[classification['abstract_id']!=fold]['true labels'].values).to(torch.int64)
        
    machine_probscores_test = torch.from_numpy(confidence[confidence['abstract_id']==fold][[machine_clf+'-A',machine_clf+'-B']].values)
    human_classification_test = torch.from_numpy(classification[classification['abstract_id']==fold]['Human'].values).to(torch.int64)
    human_confidence_test = torch.from_numpy(confidence[confidence['abstract_id']==fold]['Human'].values)
    classification_test = classification[classification['abstract_id']==fold]
    
    # Initialize model
    model = BayesianCombinationModel()
    
    # This is the training phase (labels are observed)
    logger.info(f"Fold {str(fold+1).rjust(3, ' ')}/{N} | Training...")
    model.infer(machine_probscores_train,
                human_classification_train,
                human_confidence_train,
                truelabel=truelabel_train,
                num_samples=num_samples,
                warmup_steps=warmup_steps,
                num_chains=num_chains,
                disable_progbar=True,
                group_by_chain=False)

    # This is the testing phase (labels are latent)
    logger.info(f"Fold {str(fold+1).rjust(3, ' ')}/{N} | Testing...")
    pred = model.posterior_predict(machine_probscores_test,
                                   human_classification_test,
                                   human_confidence_test)
    
    # Get predictions for this fold
    A_pred = classification_test[machine_clf].values
    B_pred = classification_test['Human'].values
    AB_pred = pred.numpy()
    true_pred = classification_test['true labels'].values
    
    # Save prediction accuracy and correlation
    tmp_pred = pd.DataFrame({'Machine':machine_clf, 'Fold': fold+1, 
                             'A': A_pred, 'B': B_pred, 'AB': AB_pred, 'true': true_pred})
    
    # Return the results
    return tmp_pred

In [ ]:
N = classification['abstract_id'].nunique()

logger.debug(f"Number of abstracts: {N}")
logger.debug(f"selected_LLMs: {selected_LLMs}")

logs_submitit = os.path.join(logsdir, 'submitit')
logger.info(f'Logs dir submitit: {logs_submitit}')
executor = submitit.SlurmExecutor(folder=logs_submitit)

executor.update_parameters(
    partition="CPU,GPU",
    time="1-00:00:00",
    mem="350G",
    cpus_per_task=48,
    comment="haico-job",
    job_name="haico-job",
    array_parallelism=300,
)

# Using array jobs with 100 parallel jobs
with executor.batch():

    # Create a list to store the jobs
    jobs = []
    
    for i in range(len(selected_LLMs)):
    
        machine_clf = classification.columns[3+i]
        logger.info(f"Classifier A: Human --- Classifier B: {machine_clf}")

        # Submit the job for each fold in parallel
        for j in range(N):
            logger.info(f"Submitting fold {j+1}/{N} for classifier {machine_clf}")
            job = executor.submit(process_cross_validation_fold, j, machine_clf, N)
            jobs.append(job)

# Wait for all jobs to complete
results = [job.result() for job in jobs]
results = pd.concat(results, ignore_index=True)

# Save raw samples and predictions
if not os.path.exists(results_path):
    os.makedirs(results_path)

for machine_clf in ['7B', '13B', '70B']:
    tmp_pred = results[results['Machine'] == machine_clf].drop(columns=['Machine'])
    tmp_pred.to_csv(os.path.join(results_path, f"brainbench_Bayesian_HM_predictions_{machine_clf}.csv"), index=False)
